# HITL ActionAgent — 데모 노트북

기존 사내 LangGraph 구조(**Router → Supervisor → members → FinalAnswerAgent**)에
**Human-In-The-Loop(HITL)** 가 적용된 `ActionAgent` 를 끼워 넣은 결과를 셀 단위로 시연한다.

- 이 노트북은 `app/` 패키지를 **import 해서 쓴다** (코드 원본은 `.py`, 노트북은 검증/시연용).
- 서버 없이 그래프를 직접 `ainvoke` 하며 `interrupt` → `Command(resume=...)` 왕복을 눈으로 본다.
- 마지막 부록에서 노트북 안에 API 서버를 띄우는 법 + Streamlit 연결법을 다룬다.

## HITL 메커니즘
| 후보 | 실존 | 판정 |
|---|---|---|
| `interrupt_before` / `interrupt_after` | ⭕ 정적 | 노드 경계에서만 멈춤 · 페이로드 못 실음 → 부적합 |
| **`interrupt()` + `Command(resume=)`** | ⭕ 동적 | **채택** — 노드/툴 내부에서 멈추고 구조화 페이로드 전달 |
| `Command.PAUSE` | ❌ 환각 | 그런 멤버 없음 (`Command` 는 resume/update/goto) |
| `__interrupt__` | △ | 트리거가 아니라 인터럽트가 **표면화되는 키**. 감지용 |

## 지켜야 하는 재개 규칙 (버전 민감)
1. interrupt 된 노드는 resume 시 **맨 위부터 재실행**된다 → interrupt 위에 side-effect 금지.
2. 한 노드의 interrupt 는 **실행 순서**로 resume 값이 매칭 → **노드당 interrupt 1개**.
3. 인터럽트 감지는 이벤트가 아니라 **`aget_state()` 상태 검사**가 정석.

## 0. 셋업

`.env` 가 없으면 `FAKE_LLM=1`(목업 LLM)로 동작하므로 사내 LLM 없이도 전부 돌아간다.

In [1]:
import os, sys, json, asyncio
from pathlib import Path

# 노트북이 notebooks/ 안에 있어도 프로젝트 루트를 import 경로에 넣는다
ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from langchain_core.messages import HumanMessage
from langgraph.types import Command

import app.config as cfg
from app._builder import build_team_graph
from app.actions.mock_db import MOCK_DB

print("ROOT      :", ROOT)
print("FAKE_LLM  :", cfg.FAKE_LLM, "(1이면 목업 LLM — 사내 endpoint 불필요)")
print("carriers  :", list(MOCK_DB["carriers"]))
print("equipment :", list(MOCK_DB["equipment"]))

ROOT      : /home/user/HITL
FAKE_LLM  : True (1이면 목업 LLM — 사내 endpoint 불필요)
carriers  : ['6PDMQ283', '3KWQ7712', '9ZXCV456', '7HITL001']
equipment : ['STK101', 'STK102', 'PHT201', 'ETC301', 'CLN501', 'DFF401']


## 1. 그래프 빌드 + 구조 확인

In [2]:
graph, checkpointer = build_team_graph()

def C(thread_id):
    """채팅 세션 하나 = thread_id 하나 (대화 맥락 유지)"""
    return {"configurable": {"thread_id": thread_id}}

print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	Router(Router)
	GeneralAgent(GeneralAgent)
	Supervisor(Supervisor)
	StatusAgent(StatusAgent)
	LocationAgent(LocationAgent)
	LogAgent(LogAgent)
	ActionAgent(ActionAgent)
	ExtractAgent(ExtractAgent)
	FinalAnswerAgent(FinalAnswerAgent)
	FinalGeneralAgent(FinalGeneralAgent)
	__end__([<p>__end__</p>]):::last
	ActionAgent --> Supervisor;
	ExtractAgent --> Supervisor;
	GeneralAgent -. &nbsp;FINISH&nbsp; .-> FinalGeneralAgent;
	GeneralAgent -.-> Supervisor;
	LocationAgent --> Supervisor;
	LogAgent --> Supervisor;
	Router -.-> GeneralAgent;
	Router -.-> Supervisor;
	StatusAgent --> Supervisor;
	Supervisor -.-> ActionAgent;
	Supervisor -.-> ExtractAgent;
	Supervisor -. &nbsp;FINISH&nbsp; .-> FinalAnswerAgent;
	Supervisor -.-> LocationAgent;
	Supervisor -.-> LogAgent;
	Supervisor -.-> StatusAgent;
	__start__ --> Router;
	FinalAnswerAgent --> __end__;
	FinalGeneralAgent --> __end__;
	classDef default f

### ActionAgent 서브그래프 내부

`ActionAgent` 는 부모에서 보면 Supervisor 밑 member 노드 하나지만, 내부는 결정적 상태 기계다.
`collect_param` 과 `confirm` 두 노드만 `interrupt()` 를 호출하고, **노드당 정확히 1개**만 둔다.

In [3]:
from app.actions.graph import build_action_graph
print(build_action_graph().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	action_entry(action_entry)
	infer_intent(infer_intent)
	param_check(param_check)
	collect_param(collect_param)
	merge_param(merge_param)
	validate(validate)
	confirm(confirm)
	execute(execute)
	finalize(finalize)
	abandon(abandon)
	needs_exit(needs_exit)
	__end__([<p>__end__</p>]):::last
	__start__ --> action_entry;
	action_entry -.-> infer_intent;
	action_entry -.-> param_check;
	collect_param --> merge_param;
	confirm -.-> abandon;
	confirm -.-> execute;
	execute --> finalize;
	infer_intent --> param_check;
	merge_param -.-> abandon;
	merge_param -.-> param_check;
	param_check -.-> abandon;
	param_check -.-> collect_param;
	param_check -.-> needs_exit;
	param_check -.-> validate;
	validate -.-> abandon;
	validate -.-> confirm;
	validate -.-> param_check;
	abandon --> __end__;
	finalize --> __end__;
	needs_exit --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fil

### 편의 함수 — 인터럽트 상태를 예쁘게 보기

In [4]:
def show_interrupt(snap):
    """체크포인터 상태에서 대기 중인 HITL 질문을 꺼내 보여준다."""
    ints = list(getattr(snap, "interrupts", None) or [])
    if not ints:
        for t in (snap.tasks or []):
            ints.extend(list(getattr(t, "interrupts", None) or []))
    if not ints:
        print("⏹  대기 중인 인터럽트 없음 (next=%s)" % (snap.next,))
        return None
    v = ints[0].value
    print("⏸  HITL 대기: type=%s" % v.get("type"))
    if v.get("field"):
        print("    묻는 파라미터: %s" % v["field"])
    print("    현재 params  : %s" % v.get("params"))
    print("    프롬프트     :")
    for ln in str(v.get("prompt", "")).split("\n"):
        print("      " + ln)
    return v

def answer(snap_cfg, text):
    """HITL 질문에 답하며 재개."""
    print("\n>>> 사용자 답변: %r\n" % text)
    return graph.ainvoke(Command(resume=text), snap_cfg)

def final_text(result):
    return result["messages"][-1].content

## 2. 시나리오 A — 필수 파라미터 수집 → 승인 → 실행

`transport`(반송요청명령)의 필수 파라미터는 `carrier_id` + `eqp_id`.
질의에 `eqp_id` 가 없으므로 **HITL 로 되묻는다.**

In [5]:
cfg_a = C("nb-A")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 반송해줘")]}, cfg_a)
v = show_interrupt(await graph.aget_state(cfg_a))

[USER] 6PDMQ283 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[TOOL param_check] enter action=transport params={'carrier_id': '6PDMQ283'}


[TOOL param_check] required=['carrier_id', 'eqp_id'] -> missing=['eqp_id']


[ACTION param_check] 미충족 -> collect 'eqp_id'


[ACTION collect_param] ⏸ interrupt field=eqp_id


⏸  HITL 대기: type=collect_param
    묻는 파라미터: eqp_id
    현재 params  : {'carrier_id': '6PDMQ283'}
    프롬프트     :
      목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


사용자가 목적지를 알려주면 → 파라미터 충족 → 검증 통과 → **두 번째 HITL(실행 승인)**

In [6]:
await answer(cfg_a, "STK102 로 보내줘")
v = show_interrupt(await graph.aget_state(cfg_a))


>>> 사용자 답변: 'STK102 로 보내줘'

[ACTION collect_param] ⏸ interrupt field=eqp_id


[ACTION collect_param] resume answer='STK102 로 보내줘'


[ACTION merge_param] enter field=eqp_id


[RESOLVER] resolve_param_answer field=eqp_id answer='STK102 로 보내줘'


[ACTION merge_param] 채움 -> params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'} action=transport


...

[ACTION validate] PASS -> confirm


[TOOL transport_confirm] enter params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION confirm] ⏸ interrupt guidance=
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)


승인하면 그때서야 `execute` 노드가 실제 액션을 수행한다 (side-effect 는 여기서만).

In [7]:
res = await answer(cfg_a, "승인")
print(final_text(res))


>>> 사용자 답변: '승인'

[TOOL transport_confirm] enter params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION confirm] ⏸ interrupt guidance=
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


[ACTION confirm] resume decision='승인' -> approve


[ACTION execute] enter action=transport params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


...

[ACTION finalize] job=TJ-20260727-0001


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


✅ 반송요청명령 실행 완료
- Job ID: TJ-20260727-0001
- 상태: DISPATCHED
- carrier_id: 6PDMQ283
- dest: STK102


## 3. 시나리오 B — 실행 거절

`dest_req`(목적지요청)는 `carrier_id` 만 있으면 되므로 곧장 승인 단계로 간다.
거절하면 **더 이상 집착하지 않고** 깨끗이 종료한다(재시도 루프 없음).

In [8]:
cfg_b = C("nb-B")
await graph.ainvoke({"messages": [HumanMessage("9ZXCV456 목적지 요청해줘")]}, cfg_b)
show_interrupt(await graph.aget_state(cfg_b))
res = await answer(cfg_b, "아니 하지마")
print(final_text(res))

[USER] 9ZXCV456 목적지 요청해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[ACTION abandon] 사용자가 실행을 거절해 명령을 종료합니다.


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


🚫 목적지요청 을(를) 실행하지 않았습니다.
- 사유: 사용자가 실행을 거절해 명령을 종료합니다.


## 4. 시나리오 C — 유효성 검증 실패 → 잘못된 파라미터만 재수집

`DFF401` 은 목업 DB 에서 **offline** 장비다. validation 이 실패하면
문제가 된 필드만 비우고 수집 루프로 되돌아간다(수렴 보장).

In [9]:
cfg_c = C("nb-C")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 를 DFF401 로 반송")]}, cfg_c)
show_interrupt(await graph.aget_state(cfg_c))   # 프롬프트에 '검증 실패' 사유가 실린다

[USER] 6PDMQ283 를 DFF401 로 반송


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[TOOL param_check] required=['carrier_id', 'eqp_id'] -> missing=['eqp_id']


[ACTION param_check] 미충족 -> collect 'eqp_id'


[ACTION collect_param] ⏸ interrupt field=eqp_id


⏸  HITL 대기: type=collect_param
    묻는 파라미터: eqp_id
    현재 params  : {'carrier_id': '6PDMQ283'}
    프롬프트     :
      검증 실패: 장비 DFF401 는 현재 오프라인이라 목적지로 지정할 수 없습니다.
      목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


{'type': 'collect_param',
 'action': 'transport',
 'field': 'eqp_id',
 'prompt': "검증 실패: 장비 DFF401 는 현재 오프라인이라 목적지로 지정할 수 없습니다.\n목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.",
 'params': {'carrier_id': '6PDMQ283'},
 'missing': ['eqp_id']}

In [10]:
await answer(cfg_c, "PHT201")
show_interrupt(await graph.aget_state(cfg_c))
res = await answer(cfg_c, "ㄱㄱ")               # 자연어 승인도 인식
print(final_text(res))


>>> 사용자 답변: 'PHT201'

[ACTION collect_param] ⏸ interrupt field=eqp_id


[ACTION collect_param] resume answer='PHT201'


[ACTION merge_param] enter field=eqp_id


[RESOLVER] resolve_param_answer field=eqp_id answer='PHT201'


[ACTION merge_param] 채움 -> params={'carrier_id': '6PDMQ283', 'eqp_id': 'PHT201'} action=transport


...

[ACTION finalize] job=TJ-20260727-0002


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


✅ 반송요청명령 실행 완료
- Job ID: TJ-20260727-0002
- 상태: DISPATCHED
- carrier_id: 6PDMQ283
- dest: PHT201


## 5. 시나리오 D — 액션 도중 탈출(취소)

모든 interrupt 지점에서 빠져나올 수 있다. resume 답변을 값/승인으로 해석하기 **전에**
취소 의도를 먼저 검사한다.

In [11]:
cfg_d = C("nb-D")
await graph.ainvoke({"messages": [HumanMessage("7HITL001 반송 부탁해")]}, cfg_d)
res = await answer(cfg_d, "아 그냥 취소해줘")
print(final_text(res))

[USER] 7HITL001 반송 부탁해


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[ACTION abandon] 사용자 요청으로 명령을 취소했습니다.


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


🚫 반송요청명령 을(를) 실행하지 않았습니다.
- 사유: 사용자 요청으로 명령을 취소했습니다.


`/chat/stop` 이 쓰는 abort 센티널도 같은 경로로 정리된다.

In [12]:
cfg_d2 = C("nb-D2")
await graph.ainvoke({"messages": [HumanMessage("7HITL001 반송해줘")]}, cfg_d2)
res = await graph.ainvoke(Command(resume={"aborted": True}), cfg_d2)   # /chat/stop 이 보내는 값
print(final_text(res))

[USER] 7HITL001 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[ACTION abandon] 사용자 요청으로 명령을 취소했습니다.


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


🚫 반송요청명령 을(를) 실행하지 않았습니다.
- 사유: 사용자 요청으로 명령을 취소했습니다.


## 6. 시나리오 E — 동료 에이전트 값 가져오기 (needs-핸드오프) ★

> ActionAgent 를 Router 에 Supervisor 와 동급으로 붙이면 다른 에이전트와 협업이 안 된다.
> 그래서 Supervisor 밑 member 로 두고, **필요한 값은 동료에게 잠깐 양보해서** 받아온다.

`"Aaa 를 Bbb 있는 위치로 반송"` → ActionAgent 가 `eqp_id` 를 스스로 알 수 없다고 판단하면
`action.needs` 를 기록하고 **interrupt 가 아니라 정상 종료**로 Supervisor 에게 양보한다.
Supervisor 가 LocationAgent 로 라우팅 → 결과를 받아 ActionAgent 재진입 → 그 파라미터는 **묻지 않는다**.

In [13]:
cfg_e = C("nb-E")
await graph.ainvoke(
    {"messages": [HumanMessage("6PDMQ283 를 9ZXCV456 있는 위치로 반송해줘")]}, cfg_e)
v = show_interrupt(await graph.aget_state(cfg_e))
print("\n→ eqp_id 를 사용자에게 묻지 않고 LocationAgent 가 채웠다:", v["params"])

[USER] 6PDMQ283 를 9ZXCV456 있는 위치로 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[ACTION validate] PASS -> confirm


[TOOL transport_confirm] enter params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION confirm] ⏸ interrupt guidance=
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)

→ eqp_id 를 사용자에게 묻지 않고 LocationAgent 가 채웠다: {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


### 로그 분석형 — LogAgent 에게 위임

`"로그 분석해서 원인 장비 피해서 …"` 처럼 **해석용 프롬프트가 필요한 요청**은
ActionAgent 가 직접 하지 않는다. "누가 필요한지"만 선언하고 LogAgent 에게 넘긴다.
(각 에이전트의 해석 프롬프트는 그 에이전트에 남는다 = 복잡도 폭발 회피)

In [14]:
cfg_f = C("nb-F")
await graph.ainvoke(
    {"messages": [HumanMessage("로그 분석해서 원인 장비 피해서 6PDMQ283 반송해줘")]}, cfg_f)
v = show_interrupt(await graph.aget_state(cfg_f))
print("\nfacts(에이전트 간 공유 팩트):")
print(json.dumps((await graph.aget_state(cfg_f)).values.get("facts", {}),
                 ensure_ascii=False, indent=2)[:800])

[USER] 로그 분석해서 원인 장비 피해서 6PDMQ283 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[ACTION validate] PASS -> confirm


[TOOL transport_confirm] enter params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION confirm] ⏸ interrupt guidance=
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)

facts(에이전트 간 공유 팩트):
{
  "log_analysis": {
    "carrier_id": "6PDMQ283",
    "combos": [
      {
        "reason": "E-STALL",
        "description": "OHT stall detected",
        "count": 3,
        "eqp": "ETC301",
        "first_t": "08:01:02"
      },
      {
        "reason": "E-PORT",
        "description": "port not ready",
        "count": 2,
        "eqp": "ETC301",
        "first_t": "08:03:40"
      }
    ],
    "cause_eqp": "ETC301",
    "recommended_dest": "STK102"
  }
}


## 7. 시나리오 G — HITL 질문에 **참조형으로** 답하기

파라미터를 물었는데 사용자가 값 대신 `"9ZXCV456 있는 위치로 채워줘"` 라고 답하는 경우.
답변 해석은 4분기다: ①취소 ②리터럴 ③참조/분석형(needs 핸드오프) ④해석불능(재질문).

In [15]:
cfg_g = C("nb-G")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 반송해줘")]}, cfg_g)
show_interrupt(await graph.aget_state(cfg_g))
await answer(cfg_g, "9ZXCV456 있는 위치로 채워줘")     # 값이 아니라 참조
v = show_interrupt(await graph.aget_state(cfg_g))
print("\n→ LocationAgent 를 경유해 자동으로 채워짐:", v["params"])

[USER] 6PDMQ283 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[ACTION validate] PASS -> confirm


[TOOL transport_confirm] enter params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION confirm] ⏸ interrupt guidance=
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)

→ LocationAgent 를 경유해 자동으로 채워짐: {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


## 8. 시나리오 H — 액션 자체가 불명확하면 액션을 묻는다

`transport` 인지 `dest_req` 인지 모를 때는 **어떤 명령인지**부터 HITL 로 확인한다.

In [16]:
cfg_h = C("nb-H")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 에 명령 실행해줘")]}, cfg_h)
show_interrupt(await graph.aget_state(cfg_h))
await answer(cfg_h, "반송으로")
show_interrupt(await graph.aget_state(cfg_h))

[USER] 6PDMQ283 에 명령 실행해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[AGENT] supervisor(fake) -> ActionAgent


...

[TOOL param_check] required=['carrier_id', 'eqp_id'] -> missing=['eqp_id']


[ACTION param_check] 미충족 -> collect 'eqp_id'


[ACTION collect_param] ⏸ interrupt field=eqp_id


⏸  HITL 대기: type=collect_param
    묻는 파라미터: eqp_id
    현재 params  : {'carrier_id': '6PDMQ283'}
    프롬프트     :
      목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


{'type': 'collect_param',
 'action': 'transport',
 'field': 'eqp_id',
 'prompt': "목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.",
 'params': {'carrier_id': '6PDMQ283'},
 'missing': ['eqp_id']}

## 9. 상태 설계 확인 — 왜 messages 밖에 두는가

`messages` 에는 최근 4턴만 남기는 리듀서가 걸려 있다. HITL 문답을 messages 에만 두면
**진행 중이던 액션이 잘려서 깨진다.** 그래서 액션 제어 상태는 `action`(교체) /
`facts`(병합) 필드에 두고, 리듀서가 건드리지 못하게 한다.

In [17]:
snap = await graph.aget_state(C("nb-H"))
sc = snap.values.get("action", {})
print("action 스크래치 (limiter 영향 없음):")
for k in ("action", "phase", "params", "missing", "pending_field",
          "collect_retries", "validate_retries", "hops"):
    if k in sc:
        print(f"  {k:16} = {sc[k]}")
print("\nmessages 개수:", len(snap.values.get("messages", [])))

action 스크래치 (limiter 영향 없음):

messages 개수: 1


## 10. 부록 — 노트북에서 API 서버 띄우고 Streamlit 붙이기

**질문: ipynb 에서 서버 띄우고 Streamlit 연결이 되나?**

- **API 서버**: 된다. 아래 셀처럼 uvicorn 을 백그라운드 스레드로 띄우면 노트북을 계속 쓰면서
  `http://localhost:8000` 으로 호출할 수 있다.
- **Streamlit**: 노트북 셀 안에서는 못 띄운다. Streamlit 은 자체 프로세스로 떠야 하므로
  **터미널에서** `streamlit run streamlit_app.py` 로 실행하고, 그 UI 가 위 API 를 호출하게 한다.

정리하면 권장 실행 형태는 이렇다.

```bash
# 터미널 1
uvicorn app.main:app --reload --port 8000
# 터미널 2
streamlit run streamlit_app.py
```

In [18]:
# 노트북 안에서 API 서버 백그라운드 기동 (선택)
import threading, time
import uvicorn
from app.main import app as fastapi_app

def _serve():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

if not any(t.name == "uvicorn-nb" for t in threading.enumerate()):
    threading.Thread(target=_serve, name="uvicorn-nb", daemon=True).start()
    time.sleep(3)

import httpx
print(httpx.get("http://127.0.0.1:8000/llm/api/health", timeout=10).json())

{'ok': True, 'fake_llm': True, 'model': 'gpt-4o', 'time': '2026-07-27T13:58:34.584873'}


### 서버에 SSE 로 질의해 보기 (Streamlit 이 하는 일과 동일)

In [19]:
import httpx, json

def ask(thread_id, query):
    """/chat/stream 을 호출하고 이벤트를 요약 출력. 반환값은 needs_input payload."""
    answer, needs, usage = "", None, None
    with httpx.Client(timeout=None) as c:
        with c.stream("POST", "http://127.0.0.1:8000/llm/api/chat/stream",
                      json={"query": query, "thread_id": thread_id}) as r:
            for line in r.iter_lines():
                if not line.startswith("data: "):
                    continue
                e = json.loads(line[6:])
                t = e["type"]
                if t == "token":
                    answer += e["text"]
                elif t == "node_enter":
                    print(f"  ▶️  {e['agent']}")
                elif t == "tool_call":
                    print(f"  🔧 {e.get('tool')} {e.get('result') or ''}")
                elif t == "agent_status":
                    print(f"  💬 {e.get('agent')}: {e.get('detail')}")
                elif t == "needs_input":
                    needs = e
                elif t == "usage":
                    usage = e
    if needs:
        print("\n⏸ HITL:", needs.get("kind"))
        print(needs.get("prompt"))
    if answer:
        print("\n💬", answer)
    if usage:
        print(f"\n🔢 {usage['total_tokens']} tok · ⏱ TTFT {usage['ttft_ms']}ms / "
              f"총 {usage['elapsed_ms']}ms (사람대기 {usage['human_wait_ms']}ms 제외 "
              f"{usage['compute_ms']}ms) · HITL {usage['hitl_rounds']}회")
    return needs

ask("nb-api", "6PDMQ283 반송해줘")

[STREAM] thread=nb-api NEW TURN <- '6PDMQ283 반송해줘'


  ▶️  Router[USER] 6PDMQ283 반송해줘



[NODE] Router entered


  💬 Router: 의도 분류 중
[AGENT] router(fake) -> supervisor


  ▶️  Supervisor
[NODE] Supervisor entered


...

[ACTION param_check] 미충족 -> collect 'eqp_id'


  🔧 param_check_tool {'missing': ['eqp_id']}
[ACTION collect_param] ⏸ interrupt field=eqp_id


[STREAM] thread=nb-api ⏸ interrupt: collect_param



⏸ HITL: collect_param
목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


{'type': 'needs_input',
 'kind': 'collect_param',
 'resume_token': '1f18977d-2454-691a-8002-9d820c4b120e',
 'action': 'transport',
 'field': 'eqp_id',
 'prompt': "목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.",
 'params': {'carrier_id': '6PDMQ283'},
 'missing': ['eqp_id'],
 'seq': 7}

In [20]:
ask("nb-api", "STK102")

[STREAM] thread=nb-api RESUME(collect_param) <- 'STK102'


  💬 HITL: 사용자 응답 수신(collect_param) — 실행 재개
[ACTION collect_param] ⏸ interrupt field=eqp_id


  ▶️  ActionAgent
[ACTION collect_param] resume answer='STK102'


[ACTION merge_param] enter field=eqp_id


[RESOLVER] resolve_param_answer field=eqp_id answer='STK102'


...

[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION confirm] ⏸ interrupt guidance=
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


[STREAM] thread=nb-api ⏸ interrupt: confirm



⏸ HITL: confirm
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


{'type': 'needs_input',
 'kind': 'confirm',
 'resume_token': '1f18977d-2454-691a-8002-9d820c4b120e',
 'action': 'transport',
 'prompt': '⚠️ 반송요청명령 실행 확인\n- 캐리어: 6PDMQ283 (현재 위치 STK101)\n- 목적지: STK102\n이 명령을 정말 실행할까요? (승인/거절)',
 'params': {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'},
 'options': ['승인', '거절'],
 'seq': 6}

In [21]:
ask("nb-api", "승인")

[STREAM] thread=nb-api RESUME(confirm) <- '승인'


  💬 HITL: 사용자 응답 수신(confirm) — 실행 재개


  ▶️  ActionAgent


[TOOL transport_confirm] enter params={'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


[TOOL transport_confirm] guidance rendered (84 chars)


...

[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered  ▶️  FinalAnswerAgent



  💬 FinalAnswerAgent: 최종 응답 생성


[saved] /home/user/HITL/logs/devLogs/2026-07/2026-07-27.jsonl



💬 ✅ 반송요청명령 실행 완료
- Job ID: TJ-20260727-0003
- 상태: DISPATCHED
- carrier_id: 6PDMQ283
- dest: STK102

🔢 163 tok · ⏱ TTFT 251ms / 총 258ms (사람대기 145ms 제외 113ms) · HITL 2회


## 11. 로그 확인

`logs/{env}/{YYYY-MM}/{YYYY-MM-DD}.jsonl` 에 턴당 1건이 기록된다.
HITL 로 여러 번의 `/chat/stream` 에 걸친 액션도 **완료 시점에 1건**으로 합산된다.

In [22]:
from app.api.routes import _get_log_dir, kst_date_str, read_log_records
p = Path(_get_log_dir()) / f"{kst_date_str()}.jsonl"
print(p)
if p.exists():
    recs = read_log_records(p)
    print(f"오늘 기록된 턴: {len(recs)}건\n")
    rec = recs[-1]
    for k in ("thread_id", "query", "route", "outcome"):
        print(f"{k:12}: {rec.get(k)}")
    print("step_history:", [s["node"] for s in rec["step_history"]])
    print("token_cost  :", json.dumps(rec["token_cost"], ensure_ascii=False))
    print("time_cost   :", json.dumps(rec["time_cost"], ensure_ascii=False))
    print("hitl        :", rec["hitl"])
else:
    print("아직 로그 없음 — 위 API 셀을 먼저 실행하세요.")

/home/user/HITL/logs/devLogs/2026-07/2026-07-27.jsonl
오늘 기록된 턴: 1건

thread_id   : nb-api
query       : 6PDMQ283 반송해줘
route       : supervisor
outcome     : complete
step_history: ['Router', 'Supervisor', 'ActionAgent', 'Supervisor', 'FinalAnswerAgent']
token_cost  : {"user_input_tokens": 4, "per_agent": {"Router": {"input": 14, "output": 7, "calls": 1}, "Supervisor": {"input": 16, "output": 7, "calls": 1}, "infer_intent": {"input": 29, "output": 20, "calls": 1}, "FinalAnswerAgent": {"input": 38, "output": 32, "calls": 1}}, "agent_input_tokens": 97, "agent_output_tokens": 66, "total_tokens": 163}
time_cost   : {"ttft_ms": 251, "elapsed_ms": 258, "human_wait_ms": 145, "compute_ms": 113}
hitl        : {'rounds': 2, 'stream_calls': 3}
